# Mission 01: Customer Purchase Data Clinic

A music store exported its purchases from three systems. The export is **real** Chinook data, but the systems added typical errors: mixed date formats, country spellings, prices with decimal commas, missing values, sign errors and re-sent records. The growth team needs a trustworthy customer summary.

**Lessons used:** 06–10 (inspection, selection, cleaning, transformation, groupby), 14 (dtypes).

The errors are added by a documented, seeded function, so you can compare your result with the original data (`clean_truth`) at the end.

In [ ]:
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent.parent / "tools"))
import datasets

def load_purchases() -> pd.DataFrame:
    """Real Chinook purchases (one row per invoice line), exported from the SQLite file."""
    with sqlite3.connect(datasets.fetch("chinook.sqlite")) as con:
        return pd.read_sql("""
            SELECT il.InvoiceLineId AS line_id, i.InvoiceId AS invoice_id, i.InvoiceDate AS invoice_date,
                   c.CustomerId AS customer_id, c.Email AS email, c.Country AS country,
                   il.TrackId AS track_id, il.UnitPrice AS unit_price, il.Quantity AS quantity
            FROM InvoiceLine il JOIN Invoice i USING (InvoiceId) JOIN Customer c USING (CustomerId)""", con)

def break_like_three_systems(df: pd.DataFrame, seed: int = 8) -> pd.DataFrame:
    """Add the typical errors of three export systems (documented, seeded, reproducible)."""
    rng = np.random.default_rng(seed)
    out = df.copy()
    n = len(out)
    dates = pd.to_datetime(out["invoice_date"])
    fmt = rng.choice(["iso", "eu", "us"], n, p=[0.6, 0.25, 0.15])            # system A: ISO, B: day.month, C: month/day
    out["invoice_date"] = np.select([fmt == "iso", fmt == "eu"],
                                    [dates.dt.strftime("%Y-%m-%d"), dates.dt.strftime("%d.%m.%Y")],
                                    dates.dt.strftime("%m/%d/%Y"))
    out["source_system"] = np.select([fmt == "iso", fmt == "eu"], ["A", "B"], "C")
    variants = {"USA": ["USA", "usa", "United States", " U.S.A. "], "United Kingdom": ["United Kingdom", "UK", "united kingdom"],
                "Brazil": ["Brazil", "Brasil", "brazil "]}
    out["country"] = [rng.choice(variants[c]) if c in variants else c for c in out["country"]]
    upper = rng.random(n) < 0.2
    out.loc[upper, "email"] = "  " + out.loc[upper, "email"].str.upper()
    out["unit_price"] = out["unit_price"].astype(object)
    out.loc[rng.random(n) < 0.05, "unit_price"] = None                       # missing prices
    comma = (rng.random(n) < 0.03) & out["unit_price"].notna()
    out.loc[comma, "unit_price"] = out.loc[comma, "unit_price"].map(lambda p: f"{p:.2f}".replace(".", ","))  # system B
    out.loc[rng.random(n) < 0.01, "quantity"] = -1                           # sign errors
    dup = out.sample(frac=0.04, random_state=seed)                           # re-sent records
    return pd.concat([out, dup]).sample(frac=1, random_state=seed).reset_index(drop=True)

clean_truth = load_purchases()
raw = break_like_three_systems(clean_truth)
print(f"{len(raw):,} raw rows from three systems")
raw.head()

## Tasks

1. **Profile.** Report the dtypes, missing values per column, the distinct country spellings and the number of exact duplicates.
2. **Parse.** Convert `invoice_date` to datetime. Each `source_system` uses one date format: use it, do not guess.
3. **Standardise.** Clean `email` (trim, lower case) and map country spellings to one name each.
4. **Repair.** Convert `unit_price` to a number (decimal commas!). Impute missing prices with the median price of the same `track_id`; fix negative quantities (decide and explain your rule).
5. **Deduplicate.** Remove re-sent records (`line_id` must be unique).
6. **Validate.** Write assertions for your rules. Compare your result with `clean_truth`: row count, total revenue, and the share of exactly matching prices.
7. **Summarise.** One row per customer: country, first and last purchase, number of invoices, revenue, average invoice value. Show the top 10 customers and plot revenue per country.

In [ ]:
# Tasks 1-7
...

## Submission Checklist

- [ ] Every cleaning rule is a named step, and you report how many rows it changed.
- [ ] Dates are parsed per source system, not guessed.
- [ ] Your revenue reconciles with `clean_truth` (or you explain the difference).
- [ ] The customer summary has one row per customer.

A solution is in `01-Customer-Purchase-Data-Clinic-solution.ipynb`. Try the mission first.